# Deploy Streamlit to GCE

This notebook automates deploying the Streamlit frontend to your GCE instance.

**Prerequisites:**
- GCE instance from step `03-gce-compute-setup/` is running
- FastAPI is running on the instance (port 5000)
- `gcloud` CLI authenticated

In [ ]:
!pip install google-cloud-compute python-dotenv

In [ ]:
import os
import subprocess
from dotenv import load_dotenv
from google.cloud import compute_v1

load_dotenv()

PROJECT_ID = os.environ['GCP_PROJECT_ID']
ZONE = os.environ.get('GCP_ZONE', 'us-central1-a')
INSTANCE_NAME = os.environ.get('GCE_INSTANCE_NAME', 'mlops-prod')

print(f'Project:  {PROJECT_ID}')
print(f'Zone:     {ZONE}')
print(f'Instance: {INSTANCE_NAME}')

In [ ]:
# Check instance status
instances_client = compute_v1.InstancesClient()
instance = instances_client.get(project=PROJECT_ID, zone=ZONE, instance=INSTANCE_NAME)

external_ip = instance.network_interfaces[0].access_configs[0].nat_i_p
print(f'Instance status: {instance.status}')
print(f'External IP:     {external_ip}')

In [ ]:
# SCP the Streamlit app to the instance
# Adjust the source path if running from a different directory
streamlit_src = '../04-local-app-development/streamlit/app.py'

scp_cmd = [
    'gcloud', 'compute', 'scp',
    streamlit_src,
    f'{INSTANCE_NAME}:~/streamlit-app/',
    '--zone', ZONE,
    '--project', PROJECT_ID,
]

print('Running:', ' '.join(scp_cmd))
result = subprocess.run(scp_cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('Error:', result.stderr)

In [ ]:
# Create firewall rule to allow port 8501 (Streamlit)
firewalls_client = compute_v1.FirewallsClient()

firewall_name = 'allow-streamlit'
existing_names = [fw.name for fw in firewalls_client.list(project=PROJECT_ID)]

if firewall_name in existing_names:
    print(f'Firewall rule {firewall_name!r} already exists — skipping.')
else:
    firewall = compute_v1.Firewall(
        name=firewall_name,
        description='Allow Streamlit traffic on port 8501',
        direction='INGRESS',
        allowed=[compute_v1.Allowed(I_p_protocol='tcp', ports=['8501'])],
        target_tags=['streamlit-server'],
        source_ranges=['0.0.0.0/0'],
    )
    op = firewalls_client.insert(project=PROJECT_ID, firewall_resource=firewall)
    op.result()  # wait for completion
    print(f'Firewall rule {firewall_name!r} created.')

In [ ]:
# Install Streamlit on the instance and run it in the background
install_and_run = (
    'mkdir -p ~/streamlit-app && '
    'pip install --quiet streamlit requests && '
    'nohup streamlit run ~/streamlit-app/app.py '
    '--server.port 8501 --server.address 0.0.0.0 '
    '> ~/streamlit.log 2>&1 &'
)

ssh_cmd = [
    'gcloud', 'compute', 'ssh', INSTANCE_NAME,
    '--zone', ZONE,
    '--project', PROJECT_ID,
    '--command', install_and_run,
]

result = subprocess.run(ssh_cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('Error:', result.stderr)

In [ ]:
print(f'Streamlit is now running at:')
print(f'  http://{external_ip}:8501')